In [9]:
#CODE TO OBTAIN THE FULL - DBDS SEQUENCES

import pandas as pd

# Load the CSV file
df = pd.read_csv('tfs.csv')

df = df.dropna(subset=['Label'])

# Create a new column 'Indexes' by concatenating values from 'Index', 'Index.1', 'Index.2', etc.
df['Indexes'] = df.apply(lambda row: ', '.join(str(row[col]) for col in df.columns if 'Index' in col and pd.notna(row[col])), axis=1)

# Create a function to remove indexes from the sequence
def remove_indexes(row):
    index_columns = [col for col in df.columns if 'Index' in col and pd.notna(row[col])]
    indexes = []
    
    for col in index_columns:
        for part in str(row[col]).split(','):
            if '-' in part:
                start, end = map(int, part.split('-'))
                indexes.append((start, end))
    
    sequence = row['Sequence']
    
    for start, end in indexes:
        sequence = sequence[:start] + sequence[end+1:]
    
    return sequence

# Apply the function to create the 'Full-DBDs' column
df['Full-DBDs'] = df.apply(remove_indexes, axis=1)

#Length of the resulting sequences
df['Full-DBDs Length'] = df['Full-DBDs'].str.len()

# Select only the desired columns
selected_columns = ['Protein', 'UniProt ID', 'Label', 'Sequence', 'Length', 'Indexes', 'Full-DBDs','Full-DBDs Length'] 

# Save the updated DataFrame to a new CSV file
df[selected_columns].to_csv('full-dbds.csv', index=False)


In [10]:
#CODE TO EXTRACT ALL COMPLIMENTARY CHUNKS OUTSIDE OF THE DBDS ORDERED BY INDEX

df = pd.read_csv('tfs.csv')

df = df.dropna(subset=['Label'])

# Create a new column 'Indexes' by concatenating values from 'Index', 'Index.1', 'Index.2', etc.
df['Indexes'] = df.apply(lambda row: ', '.join(str(row[col]) for col in df.columns if 'Index' in col and pd.notna(row[col])), axis=1)

# Calculate 'CIndexes' based on the logic provided
def calculate_cindexes(row):
    if pd.notna(row['Sequence']):
        sequence_length = len(row['Sequence'])
        cindexes_list = []
        
        if pd.notna(row['Indexes']) and row['Indexes'] != '':
            indexes_ranges = [list(map(int, index_range.split('-'))) for index_range in row['Indexes'].split(', ')]
            cindexes_list.append(f"1-{indexes_ranges[0][0] - 1}")

            for i in range(len(indexes_ranges) - 1):
                start, end = indexes_ranges[i][1] + 1, indexes_ranges[i+1][0] - 1
                cindexes_list.append(f"{start}-{end}")
            
            # Handling the last range and the remaining length
            last_start, last_end = indexes_ranges[-1][1] + 1, sequence_length
            cindexes_list.append(f"{last_start}-{last_end}")
        
        return ', '.join(cindexes_list)

    return ""

df['CIndexes'] = df.apply(calculate_cindexes, axis=1)

# Function to extract sequence chunks based on CIndexes
def extract_sequence_chunks(row):
    sequence_chunks = []
    if pd.notna(row['CIndexes']) and row['CIndexes'] != '':
        chunks = [row['Sequence'][int(start)-1:int(end)] for start, end in (index_range.split('-') for index_range in row['CIndexes'].split(', '))]
        sequence_chunks.extend(chunks)

    return sequence_chunks

# Apply the function to create a new column with a list of sequence chunks
df['SequenceChunksList'] = df.apply(extract_sequence_chunks, axis=1)

# Get the maximum number of chunks in any row
max_chunks = max(len(chunks) for chunks in df['SequenceChunksList'])

# Create columns dynamically based on the maximum number of chunks
chunk_columns = [f'Chunk{i+1}' for i in range(max_chunks)]

# Expand the list of chunks into separate columns
df[chunk_columns] = pd.DataFrame(df['SequenceChunksList'].tolist(), index=df.index)

# Select only the desired columns
selected_columns = ['Protein', 'UniProt ID', 'Label', 'Sequence','Length', 'Indexes', 'CIndexes'] + chunk_columns

# Save the updated DataFrame to a new CSV file
df[selected_columns].to_csv('ordered_sequences_without_dbds.csv', index=False)


In [11]:
#CODE TO ORDER CHUNKS AS N-, C-TERMINUS AND REST OF THE CHUNKS ORDERED BY LENGTH

# Load the CSV file
df = pd.read_csv('ordered_sequences_without_dbds.csv')

# Select only the Chunk columns starting from Chunk2
chunk_columns = [col for col in df.columns if col.startswith('Chunk')]

# Function to rearrange the order of chunks individually per row
def rearrange_chunks(row):
    chunks = [row[col] for col in reversed(chunk_columns)]
    
    # Place the last chunk in Chunk2 and order the rest from longest to shortest
    row['Chunk2'] = chunks[-1]
    sorted_chunks = sorted([chunk for chunk in chunks[:-1] if pd.notna(chunk)], key=len, reverse=True)
    
    # Update the remaining Chunk columns
    for i, chunk in enumerate(sorted_chunks, start=3):
        row[f'Chunk{i}'] = chunk
    
    return row

# Apply the function to each row to rearrange the order of chunks
df = df.apply(rearrange_chunks, axis=1)

# Select only the desired columns
selected_columns = ['Protein', 'UniProt ID', 'Label', 'Sequence', 'Length', 'Indexes', 'CIndexes'] + chunk_columns

# Save the updated DataFrame to a new CSV file
df[selected_columns].to_csv('out_rearranged_individual.csv', index=False)
